# 02 – FDG PET/CT Segmentation Practice

**SNMMI AI Certificate – Hands-On Module**

In this notebook you will:
- Download an FDG PET/CT 2-D patch dataset
- Train a compact 2-D U-Net from scratch
- Complete **three focused tasks** marked with ✏️

**Student TODOs:**
1. ✏️ **Implement Dice loss** (Section 4)
2. ✏️ **Implement at least one data augmentation** (Section 2)
3. ✏️ **Compute Dice & total tumor volume on validation cases** (Section 7)

> **Run cells top-to-bottom on a fresh Colab runtime.**
> The notebook runs end-to-end with the default working code (BCE loss, no augmentation) before you edit anything.

---

**Dataset citation:**

> Gatidis, S., Kuestner, T., Ingrisch, M., Hepp, T., Frueh, M., Nikolaou, K.,
> La Fougere, C., Pfannenberg, C., Fabritius, M., Jeblick, K., Schachtner, B.,
> Dexl, J., Wesp, P., Mittermeier, A., Unterrainer, L., Sheikh, G., Boening, G.,
> Brendel, M., Ricke, J., … Cyran, C. (2025).
> *PSMA-FDG-PET-CT-Lesions*.
> University of Tuebingen, Ludwig-Maximilians-University Munich.
> [https://doi.org/10.57754/FDAT.0zs4c-89f12](https://doi.org/10.57754/FDAT.0zs4c-89f12)
>
> License: **CC BY-NC 4.0** – non-commercial use only.

## 0 · Setup & Config

In [ ]:
%pip install -q numpy matplotlib tqdm scikit-learn
# torch and torchvision are pre-installed on Colab

import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
from tqdm import tqdm

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Hyperparameters ───────────────────────────────────────────────────────────
EPOCHS     = 3       # keep ≤ 3 for free Colab
BATCH_SIZE = 16
LR         = 1e-3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {DEVICE}")

## 1 · Download & Unzip FDG Sample Data

In [ ]:
# ── Paste your GitHub Release URL for the FDG zip here ───────────────────────
DATA_URL = "https://github.com/ZekunLi-Zeke/snmmi-ai-hands-on-segmentation/releases/download/v0.1/fdg_sample.zip"
# ─────────────────────────────────────────────────────────────────────────────

import pathlib, urllib.request, zipfile

DATA_DIR = pathlib.Path("./data/fdg")
DATA_DIR.mkdir(parents=True, exist_ok=True)

zip_path = pathlib.Path("/tmp/fdg_sample.zip")

if not any(DATA_DIR.rglob("*.npz")):
    if not zip_path.exists():
        print("Downloading FDG sample data …")
        urllib.request.urlretrieve(DATA_URL, zip_path)
        print("Download complete.")
    print("Unzipping …")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DATA_DIR)
    print("Done.")
else:
    print("Data already present.")

npz_files = sorted(DATA_DIR.rglob("*.npz"))
assert len(npz_files) > 0, "No .npz file found – check DATA_URL and re-run."
print(f"Found: {[f.name for f in npz_files]}")

data = np.load(npz_files[0])
print(f"Keys: {list(data.keys())}")
for k in data:
    print(f"  {k}: shape={data[k].shape}  dtype={data[k].dtype}")

## 2 · Load Data & Visualise

### ✏️ Your Turn – Task 2: Implement at Least One Augmentation

Inside `PatchDataset.__getitem__`, find the `# TODO` block and implement
**at least one** augmentation (e.g., random horizontal flip, vertical flip,
90° rotation). The augmentation should be applied consistently to PET, CT,
and mask arrays. Set `AUGMENT = True` after implementing.

**Hint – horizontal flip:**
```python
pet  = pet[:, ::-1].copy()
ct   = ct[:, ::-1].copy()
mask = mask[:, ::-1].copy()
```

In [ ]:
# ── Augmentation flag (default OFF; flip in Task 2) ───────────────────────
AUGMENT = False   # ← set to True after implementing your augmentation

class PatchDataset(Dataset):
    """2-D PET/CT patch dataset from a .npz file.

    Expected .npz keys
    ------------------
    pet  : (N, 64, 64) float32 – SUV values
    ct   : (N, 64, 64) float32 – HU values (normalised)
    mask : (N, 64, 64) float32 – binary lesion mask {0, 1}
    """

    def __init__(self, npz_path, augment=False):
        data = np.load(npz_path)
        self.pet    = data["pet"].astype(np.float32)
        self.ct     = data["ct"].astype(np.float32)
        self.mask   = data["mask"].astype(np.float32)
        self.augment = augment

    def __len__(self):
        return len(self.pet)

    def __getitem__(self, idx):
        pet  = self.pet[idx].copy()
        ct   = self.ct[idx].copy()
        mask = self.mask[idx].copy()

        # ── TODO (Task 2): implement at least one augmentation ─────────────
        # Apply the augmentation to pet, ct, AND mask consistently.
        # Example: random horizontal flip with 50 % probability.
        # if self.augment and random.random() > 0.5:
        #     pet  = ...  # flip pet
        #     ct   = ...  # flip ct
        #     mask = ...  # flip mask
        # ───────────────────────────────────────────────────────────────────

        x = np.stack([pet, ct], axis=0)  # (2, 64, 64)
        y = mask[np.newaxis]              # (1, 64, 64)
        return torch.from_numpy(x), torch.from_numpy(y)


npz_path = sorted(DATA_DIR.rglob("*.npz"))[0]
full_ds  = PatchDataset(npz_path, augment=AUGMENT)
n_total  = len(full_ds)
n_val    = max(1, int(0.2 * n_total))
n_train  = n_total - n_val

train_ds, val_ds = random_split(
    full_ds, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)
print(f"Dataset: {n_total} patches  |  train {n_train}  val {n_val}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# Visualise a few patches
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for col in range(4):
    x, y = full_ds[col]
    axes[0, col].imshow(x[0].numpy(), cmap="hot");   axes[0, col].set_title(f"PET #{col}")
    axes[1, col].imshow(x[1].numpy(), cmap="gray");  axes[1, col].set_title(f"CT #{col}")
    axes[2, col].imshow(y[0].numpy(), cmap="gray");  axes[2, col].set_title(f"Mask #{col}")
for ax in axes.flat:
    ax.axis("off")
plt.suptitle("Sample patches (PET / CT / Mask)", fontsize=13)
plt.tight_layout()
plt.show()

## 3 · Define Compact 2-D U-Net

In [ ]:
class DoubleConv(nn.Module):
    """Two consecutive 3×3 conv → BN → ReLU blocks."""

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet2D(nn.Module):
    """Compact 2-D U-Net with skip connections (4 resolution levels)."""

    def __init__(self, in_ch=2, out_ch=1, features=16):
        super().__init__()
        f = features
        self.enc1 = DoubleConv(in_ch, f)
        self.enc2 = DoubleConv(f,     f * 2)
        self.enc3 = DoubleConv(f * 2, f * 4)
        self.pool  = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(f * 4, f * 8)
        self.up3  = nn.ConvTranspose2d(f * 8, f * 4, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(f * 8, f * 4)
        self.up2  = nn.ConvTranspose2d(f * 4, f * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(f * 4, f * 2)
        self.up1  = nn.ConvTranspose2d(f * 2, f,     kernel_size=2, stride=2)
        self.dec1 = DoubleConv(f * 2, f)
        self.head = nn.Conv2d(f, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b  = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b),  e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)   # (B, 1, H, W) raw logits


model    = UNet2D(in_ch=2, out_ch=1, features=16).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"U-Net parameters: {n_params:,}")

## 4 · Define Loss & Optimizer

### ✏️ Your Turn – Task 1: Implement Dice Loss

The default loss below is **BCE only**.  Your task:

1. Complete the `dice_loss()` function body (the formula is given in comments).
2. Set `LOSS_TYPE = "dice_bce"` to use your Dice + BCE combined loss.

**Dice loss formula:**

$$\text{Dice Loss} = 1 - \frac{2 \sum (p \cdot g) + \epsilon}{\sum p + \sum g + \epsilon}$$

where $p$ = predicted probabilities (after sigmoid), $g$ = ground truth, $\epsilon$ = smoothing constant.

In [ ]:
# ── Task 1: set loss type ─────────────────────────────────────────────
LOSS_TYPE = "bce"   # ← change to "dice_bce" after implementing dice_loss()
# ───────────────────────────────────────────────────────────────────

def dice_loss(logits, target, smooth=1.0):
    """Differentiable soft Dice loss (applied after sigmoid).

    Steps:
      1. Apply sigmoid to logits to get predicted probabilities.
      2. Compute intersection: sum of (pred * target) over spatial dims (2, 3).
      3. Compute denominator: sum of pred + sum of target over spatial dims.
      4. Return  1 - mean( (2 * intersection + smooth) / (denominator + smooth) )
    """
    # ── TODO (Task 1): implement Dice loss ───────────────────────────────
    # pred  = torch.sigmoid(logits)
    # inter = ...
    # denom = ...
    # return 1.0 - ...
    raise NotImplementedError("Implement dice_loss before setting LOSS_TYPE='dice_bce'")
    # ───────────────────────────────────────────────────────────────────


def get_loss_fn(loss_type):
    if loss_type == "dice_bce":
        def fn(logits, target):
            return dice_loss(logits, target) + nn.functional.binary_cross_entropy_with_logits(logits, target)
        return fn
    else:
        # Default: BCE only
        return nn.functional.binary_cross_entropy_with_logits


loss_fn   = get_loss_fn(LOSS_TYPE)
optimizer = optim.Adam(model.parameters(), lr=LR)
print(f"Loss: {LOSS_TYPE}  |  Optimizer: Adam  |  LR: {LR}")

## 5 · Training Loop (2–3 Epochs)

In [ ]:
def dice_score(logits, target, threshold=0.5):
    """Dice coefficient metric (non-differentiable)."""
    pred  = (torch.sigmoid(logits) > threshold).float()
    inter = (pred * target).sum(dim=(2, 3))
    denom = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    return ((2.0 * inter + 1.0) / (denom + 1.0)).mean().item()


history = {"train_loss": [], "val_dice": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [train]", leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * x.size(0)
    epoch_loss /= n_train

    model.eval()
    val_dice_sum = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            val_dice_sum += dice_score(model(x), y) * x.size(0)
    val_dice = val_dice_sum / n_val

    history["train_loss"].append(epoch_loss)
    history["val_dice"].append(val_dice)
    print(f"Epoch {epoch:>2}: train loss={epoch_loss:.4f}  val Dice={val_dice:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(range(1, EPOCHS + 1), history["train_loss"], marker="o")
ax1.set(title="Train Loss", xlabel="Epoch", ylabel="Loss")
ax2.plot(range(1, EPOCHS + 1), history["val_dice"], marker="o", color="orange")
ax2.set(title="Validation Dice", xlabel="Epoch", ylabel="Dice")
plt.tight_layout()
plt.show()

## 6 · Visualise Predictions

In [ ]:
model.eval()
n_show = min(4, len(val_ds))
fig, axes = plt.subplots(n_show, 4, figsize=(14, 3.5 * n_show))

with torch.no_grad():
    for row in range(n_show):
        x, y  = val_ds[row]
        logit = model(x.unsqueeze(0).to(DEVICE))
        pred  = (torch.sigmoid(logit) > 0.5).float().cpu().squeeze().numpy()

        axes[row, 0].imshow(x[0].numpy(), cmap="hot");  axes[row, 0].set_title("PET")
        axes[row, 1].imshow(x[1].numpy(), cmap="gray"); axes[row, 1].set_title("CT")
        axes[row, 2].imshow(y[0].numpy(), cmap="gray"); axes[row, 2].set_title("GT Mask")
        axes[row, 3].imshow(pred,          cmap="gray"); axes[row, 3].set_title("DL Prediction")

for ax in axes.flat:
    ax.axis("off")
plt.suptitle("Predictions vs Ground Truth", fontsize=14)
plt.tight_layout()
plt.show()

## 7 · ✏️ Your Turn – Task 3: Compute Dice & Total Tumor Volume

Complete the cell below:
1. Compute **Dice** on each validation patch.
2. Compute the **total tumor volume** (total lesion pixel count summed across all
   validation patches) for both GT and DL predictions.

Fill in the `# TODO` blocks in the next cell.

After finishing this task, go back and also try:
- **Task 1** – implement `dice_loss()` in Section 4, set `LOSS_TYPE = "dice_bce"`,
  and re-run from Section 4 onward.
- **Task 2** – implement an augmentation in Section 2, set `AUGMENT = True`,
  and re-run from Section 2 onward.

Compare the validation Dice before and after each change.

In [ ]:
# ── Task 3: Compute Dice & total tumor volume ─────────────────────────────
model.eval()

all_dice  = []
gt_areas  = []   # per-patch GT lesion pixel count
dl_areas  = []   # per-patch predicted lesion pixel count

with torch.no_grad():
    for i in range(len(val_ds)):
        x, y    = val_ds[i]
        gt_mask = y[0].numpy()                        # (64, 64)
        logit   = model(x.unsqueeze(0).to(DEVICE))    # (1, 1, 64, 64)
        dl_pred = (torch.sigmoid(logit) > 0.5).float().cpu().squeeze().numpy()

        # ── TODO 3a: compute Dice for this patch and append to all_dice ───
        # Dice = (2 * intersection + smooth) / (sum_pred + sum_gt + smooth)
        # smooth = 1.0 avoids division by zero.
        # inter  = (dl_pred * gt_mask).sum()
        # dice   = ...
        # all_dice.append(dice)
        pass

        # ── TODO 3b: compute tumor area (pixel count) for GT and prediction ─
        # gt_areas.append(int(gt_mask.sum()))
        # dl_areas.append(int(dl_pred.sum()))
        pass

# ── TODO 3c: print mean Dice and total tumor volume ─────────────────────
# Total tumor volume = sum of all lesion pixels across validation patches.
# print(f"Mean Dice:             {np.mean(all_dice):.3f}")
# print(f"Total GT tumor volume: {sum(gt_areas)} px")
# print(f"Total DL tumor volume: {sum(dl_areas)} px")

print("Task 3: fill in the TODO blocks above and re-run this cell.")

## Expected Outputs

After completing all three tasks you should observe:

| Metric | Typical range after 3 epochs (free Colab CPU) |
|--------|-----------------------------------------------|
| Val Dice (BCE only) | 0.40 – 0.65 |
| Val Dice (Dice+BCE) | 0.50 – 0.75 |
| Total DL tumor volume | Close to total GT tumor volume |

- Enabling augmentation may slightly improve Dice after only 3 epochs.
- With `Dice+BCE` loss, training curves tend to be smoother.
- Total DL tumor volume should be roughly similar to GT tumor volume.

> 🏁 Well done! You have trained a U-Net on real PET/CT data,
> implemented a segmentation loss, added augmentation, and evaluated
> performance using Dice and tumor volume.